In [1]:
import os
os.chdir('/kaggle/working')
!rm -rf /kaggle/working/repo
!pip uninstall ultralytics -y -q
!git clone https://github.com/AadeeshRS/road-damage-detection-thesis.git /kaggle/working/repo
!pip install -e /kaggle/working/repo/ultralytics -q
!pip install sahi -q
print("Setup done!")

Cloning into '/kaggle/working/repo'...
remote: Enumerating objects: 1511, done.
remote: Counting objects: 100% (1115/1115), done.
remote: Compressing objects: 100% (862/862), done.
remote: Total 1511 (delta 281), reused 1062 (delta 237), pack-reused 396 (from 3)
Receiving objects: 100% (1511/1511), 478.42 MiB | 41.22 MiB/s, done.
Resolving deltas: 100% (342/342), done.
Updating files: 100% (1192/1192), done.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.6/148.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.7 MB/s eta 0:00:00
Setup done!


In [2]:
import sys
# Clear cache so it uses the custom repo
for key in list(sys.modules.keys()):
    if 'ultralytics' in key:
        del sys.modules[key]
sys.path.insert(0, '/kaggle/working/repo/ultralytics')

import json
import glob
from PIL import Image
from tqdm import tqdm
import pandas as pd
from sahi.predict import get_sliced_prediction
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from sahi import AutoDetectionModel


In [6]:
WEIGHTS_PATH = "/kaggle/input/datasets/aadeeshranjan/hybrid-india/best.pt"

COCO_GT_PATH = "/kaggle/working/india_instances_val.json"
COCO_PRED_PATH = "/kaggle/working/india_predictions.json"
DATASET_PATH = "/kaggle/input/datasets/aliabdelmenam/rdd-2022/RDD_SPLIT"

categories = [
    {"id": 1, "name": "longitudinal_crack"},
    {"id": 2, "name": "transverse_crack"},
    {"id": 3, "name": "alligator_crack"},
    {"id": 4, "name": "other_corruption"},
    {"id": 5, "name": "pothole"},
]

# Re-create val_india.txt
val_india = [f for f in glob.glob(f"{DATASET_PATH}/val/images/*.jpg") if os.path.basename(f).startswith("India_")]
with open("/kaggle/working/val_india.txt", "w") as f:
    f.write("\n".join(val_india))
    
print(f"Loaded {len(val_india)} India validation images.")


Loaded 1172 India validation images.


In [7]:
images = []
annotations = []
annotation_id = 1

print("Converting YOLO annotations to COCO format...")
for image_path in tqdm(val_india):
    image_name = os.path.basename(image_path)
    label_path = image_path.replace("/images/", "/labels/").replace(".jpg", ".txt")
    
    width, height = Image.open(image_path).size
    image_id = image_name.replace(".jpg", "")
    
    images.append({
        "id": image_id,
        "file_name": image_name,
        "width": width,
        "height": height
    })
    
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                cls, xc, yc, bw, bh = map(float, line.split())
                x = (xc - bw / 2) * width
                y = (yc - bh / 2) * height
                w = bw * width
                h = bh * height
                
                annotations.append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(cls) + 1,
                    "bbox": [float(x), float(y), float(w), float(h)],
                    "area": float(w * h),
                    "iscrowd": 0
                })
                annotation_id += 1

coco_gt = {"images": images, "annotations": annotations, "categories": categories}
with open(COCO_GT_PATH, "w") as f:
    json.dump(coco_gt, f)
print(f"Ground Truth COCO saved: {len(annotations)} annotations.")


Converting YOLO annotations to COCO format...


100%|██████████| 1172/1172 [00:01<00:00, 646.29it/s]

Ground Truth COCO saved: 1249 annotations.


In [8]:
print("Running SAHI Inference for mAP Evaluation...")

detection_model = AutoDetectionModel.from_pretrained(
    model_type='yolov8',
    model_path=WEIGHTS_PATH,
    confidence_threshold=0.25,
    device="cuda:0"
)

predictions = []

for image_path in tqdm(val_india):
    image_name = os.path.basename(image_path)
    image_id = image_name.replace(".jpg", "")
    
    result = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )
    
    for pred in result.object_prediction_list:
        x, y, w, h = pred.bbox.to_xywh()
        predictions.append({
            "image_id": image_id,
            "category_id": int(pred.category.id) + 1,
            "bbox": [float(x), float(y), float(w), float(h)],
            "score": float(pred.score.value)
        })

with open(COCO_PRED_PATH, "w") as f:
    json.dump(predictions, f)
print(f"Predictions saved: {len(predictions)} objects detected.")


Running SAHI Inference for mAP Evaluation...


100%|██████████| 1172/1172 [02:53<00:00,  6.74it/s]

Predictions saved: 591 objects detected.


In [9]:
print("Calculating SAHI mAP Metrics...")
coco_gt_eval = COCO(COCO_GT_PATH)
coco_dt_eval = coco_gt_eval.loadRes(COCO_PRED_PATH)

coco_eval = COCOeval(coco_gt_eval, coco_dt_eval, iouType="bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

# SAVE TO CSV
summary = {
    "Precision (mAP50)": coco_eval.stats[1],
    "mAP50-95": coco_eval.stats[0],
    "Recall (AR@100)": coco_eval.stats[8],
}
df = pd.DataFrame([summary])
df.to_csv("/kaggle/working/sahi_india_metrics.csv", index=False)
display(df)


Calculating SAHI mAP Metrics...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.40s).
Accumulating evaluation results...
DONE (t=0.07s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.046
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.115
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.025
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.023
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.025
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.045
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.079
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.084
 Average Recall     (AR) @[ IoU=0

,Precision (mAP50),mAP50-95,Recall (AR@100)
0,0.11516,0.045695,0.08399


In [10]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO
from sahi.utils.cv import visualize_object_predictions

print("Calculating standard YOLO detections to find % improvement...")

# Load standard YOLO model
model = YOLO(WEIGHTS_PATH)

candidate_images = []
total_yolo = 0
total_sahi = 0

for image_path in tqdm(val_india):
    # Standard YOLO inference
    yolo_count = len(model(image_path, verbose=False)[0].boxes)
    total_yolo += yolo_count

    # SAHI Sliced Inference
    sahi_result = get_sliced_prediction(
        image_path,
        detection_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.2,
        overlap_width_ratio=0.2,
        verbose=0
    )
    sahi_count = len(sahi_result.object_prediction_list)
    total_sahi += sahi_count

    gain = sahi_count - yolo_count

    candidate_images.append({
        "image_path": image_path,
        "yolo": yolo_count,
        "sahi": sahi_count,
        "gain": gain,
        "sahi_result": sahi_result
    })

# Print Overall Improvement
improvement = ((total_sahi - total_yolo) / total_yolo) * 100 if total_yolo > 0 else 0
print(f"\n--- DETECTION SUMMARY ---")
print(f"Standard YOLO Detections: {total_yolo}")
print(f"SAHI Detections: {total_sahi}")
print(f"Improvement: +{improvement:.2f}%\n")


Calculating standard YOLO detections to find % improvement...


100%|██████████| 1172/1172 [03:26<00:00,  5.67it/s]


--- DETECTION SUMMARY ---
Standard YOLO Detections: 465
SAHI Detections: 591
Improvement: +27.10%



In [11]:
output_dir = "/kaggle/working/sahi_india_top_gains"
os.makedirs(output_dir, exist_ok=True)

# Sort to find the images with the biggest gain
candidate_images = sorted(candidate_images, key=lambda x: x["gain"], reverse=True)
top_images = candidate_images[:5]

print("Generating Top 5 visual comparisons...")
for idx, item in enumerate(top_images, start=1):
    image_path = item["image_path"]
    
    # Get YOLO visualization
    yolo_img = model(image_path, verbose=False)[0].plot()

    # Get SAHI visualization
    original = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    vis = visualize_object_predictions(
        image=original,
        object_prediction_list=item["sahi_result"].object_prediction_list,
        output_dir=None
    )
    sahi_img = vis["image"]

    # Plot Side-by-Side
    fig, axes = plt.subplots(1, 2, figsize=(12,6))
    axes[0].imshow(cv2.cvtColor(yolo_img, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"YOLO ({item['yolo']} objects)")
    axes[0].axis("off")

    axes[1].imshow(sahi_img)
    axes[1].set_title(f"SAHI ({item['sahi']} objects)")
    axes[1].axis("off")

    plt.tight_layout()
    plt.savefig(f"{output_dir}/top_gain_{idx}.png", dpi=300)
    plt.close()

print(f"Top 5 comparisons saved to {output_dir}")


Generating Top 5 visual comparisons...
Top 5 comparisons saved to /kaggle/working/sahi_india_top_gains


In [12]:
!zip -r /kaggle/working/sahi_india_top_gains.zip /kaggle/working/sahi_india_top_gains


  adding: kaggle/working/sahi_india_top_gains/ (stored 0%)
  adding: kaggle/working/sahi_india_top_gains/top_gain_4.png (deflated 2%)
  adding: kaggle/working/sahi_india_top_gains/top_gain_3.png (deflated 2%)
  adding: kaggle/working/sahi_india_top_gains/top_gain_5.png (deflated 2%)
  adding: kaggle/working/sahi_india_top_gains/top_gain_1.png (deflated 2%)
  adding: kaggle/working/sahi_india_top_gains/top_gain_2.png (deflated 2%)
